# Chapter 13 &mdash; The Context Window Is the Memory

**Concept 15 of the Chapter 13 decomposition:** *The Context Window Is the Memory*

Work out on paper where the accuracy must collapse, then measure it there &mdash; and watch five times the training fail to move it.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Context-Window-Is-The-Memory/Concept-Context-Window-Is-The-Memory.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The previous notebook found a wall. This one derives its position before measuring it,
which is a better class of result than a plot.

**The arithmetic.** In $w\#w$ the second-half symbol at offset $j$ is a copy of the
first-half symbol at offset $j$. Count the gap between them: the first half occupies
positions $0 \ldots |w|-1$, the separator sits at $|w|$, and the second half runs
from $|w|+1$. So the symbol at position $|w|+1+j$ is a copy of the one at position
$j$, exactly

$$(|w|+1+j) - j \;=\; |w|+1$$

positions back &mdash; **the same distance for every $j$**.

A model with a window of $k$ symbols sees that far back exactly when $|w|+1 \le k$.
So the prediction is not "accuracy degrades as words get longer". It is:

> accuracy is high for $|w| \le k-1$, and sits at **exactly one half** for every
> $|w| \ge k$, with a cliff and not a slope between them.

Predict the table before you run it. Then the control that matters: train the small
model far longer and see whether the floored columns move at all.

## 2. Definitions

### The model, the training loop

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

&nbsp;

In [ ]:
def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

### The corpus, the measurement, one training run

In [ ]:
# --- the corpus.  GENERATED, not filtered --------------------------------
# Strings of the form w#w are exponentially rare, so the enumerate-and-test
# recipe of Chapters 6 and 12 would find almost nothing.  Write the w down
# and double it instead.
#
# `per` words of EVERY length matters more than it looks.  Take all of them
# and the length-6 words outnumber the length-1 words 32 to 1, so every
# per-length number below would be computed from a handful of positions --
# that is how you get a 100% that is one lucky guess.  Balance first.
import random
from itertools import product

SYM = {'0': 0, '1': 1, '#': 2}

def corpus(nmax=6, per=64, seed=0, mirror=False):
    rnd, out = random.Random(seed), []
    for n in range(1, nmax + 1):
        ws = [''.join(w) for w in product('01', repeat=n)]
        for i in range(per):
            w = ws[i % len(ws)]
            out.append(w + '#' + (w[::-1] if mirror else w))
    rnd.shuffle(out)
    return out

def encode(strings):
    """the flat token sequence, plus for each position which second-half
       symbol it is: its |w| and its offset j into the second half."""
    seq, wlen, pos = [], [], []
    for s in strings:
        n = s.index('#')
        for i, c in enumerate(s):
            seq.append(SYM[c])
            wlen.append(n if i > n else None)
            pos.append(i - n - 1 if i > n else None)
    return seq, wlen, pos

# --- accuracy on the second half, grouped by whatever you like -----------
# The model is asked for the single most likely next token (argmax, not a
# sample), and it is asked only at positions that COPY -- the second half.
# 50% is the coin: the two bits are equally likely when you cannot see the
# original.
def accuracy_by(gpt, k, seq, tag):
    idx = [i for i in range(k, len(seq)) if tag[i] is not None]
    hit, tot = {}, {}
    for b in range(0, len(idx), 8192):
        chunk = idx[b:b + 8192]
        X = torch.tensor([seq[i - k:i] for i in chunk], dtype=torch.long)
        pred = gpt(X).argmax(-1).tolist()
        for i, p in zip(chunk, pred):
            t = tag[i]
            tot[t] = tot.get(t, 0) + 1
            hit[t] = hit.get(t, 0) + (p == seq[i])
    return {t: (hit[t] / tot[t], tot[t]) for t in sorted(tot)}

def table(rows, keys, label):
    print('%-9s' % label + ''.join('%7s' % k for k in keys))
    for name, acc in rows:
        print('%-9s' % name + ''.join('%6.0f%%' % (100 * acc[k][0])
                                      for k in keys))
    print('%-9s' % 'samples' + ''.join('%7d' % rows[0][1][k][1] for k in keys))

# --- one training run, with every knob in the signature ------------------
def train_at(seq, k, iters=300, n_embd=16, seed=1337, quiet=False):
    X, Y = make_XY(seq, k)
    config = GPTConfig(block_size=k, vocab_size=3, n_layer=4, n_head=4,
                       n_embd=n_embd, bias=False)
    torch.manual_seed(seed)
    gpt = GPT(config)
    losses = train_gpt(gpt, X, Y, iters=iters,
                       every=iters if quiet else iters // 4)
    return gpt, losses[-1]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;14.&nbsp;A Transformer on a Language That Needs a Tape](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Karpathy-GPT-On-A-Jove-TM/Concept-Karpathy-GPT-On-A-Jove-TM.ipynb) &nbsp;&middot;&nbsp; [**Chapter 13** index](https://github.com/ganeshutah/Jove/blob/master/Chapter13-TM/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;16.&nbsp;Copy Versus Mirror: the Hierarchy Reversed](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Copy-Versus-Mirror/Concept-Copy-Versus-Mirror.ipynb)&nbsp;&rarr;

---

## 3. Tests

One corpus for every model below.

In [ ]:
strings = corpus(nmax=6, per=64)
seq, wlen, pos = encode(strings)
print('%d strings, %d symbols, balanced across lengths 1..6'
      % (len(strings), len(seq)))

**The prediction, written down first.**

In [ ]:
for k in (4, 6, 9):
    fits = [n for n in range(1, 7) if n + 1 <= k]
    floor = [n for n in range(1, 7) if n + 1 > k]
    print('k=%-2d  window reaches the copy for |w| in %-12s  coin for %s'
          % (k, str(fits), str(floor) if floor else 'nothing'))

**Now measure.** Same corpus, same iterations, three window sizes.

In [ ]:
rows = []
for k in (4, 6, 9):
    gpt, fl = train_at(seq, k, iters=300, quiet=True)
    acc = accuracy_by(gpt, k, seq, wlen)
    rows.append(('k=%d' % k, acc))
    print('k=%-2d final loss %.4f' % (k, fl))
print()
table(rows, list(range(1, 7)), '|w| =')

Compare that table with the prediction cell, column by column.

In [ ]:
print("k=4  : floored from |w|=4 on.        predicted |w| >= 4.")
print("k=6  : floored at |w|=6.             predicted |w| >= 6.")
print("k=9  : nothing floored at all.       predicted nothing floored.")
print()
print("The wall is not approximately in the right place.  It is exactly")
print("where counting the positions said it would be, and it moves when")
print("you move k, by the amount the arithmetic says.")

**The control.** Is 50% just undertraining? Give the small model five times the budget.

In [ ]:
rows = []
for k in (4, 9):
    gpt, fl = train_at(seq, k, iters=1500, quiet=True)
    rows.append(('k=%d' % k, accuracy_by(gpt, k, seq, wlen)))
    print('k=%-2d  1500 iterations, final loss %.4f' % (k, fl))
print()
table(rows, list(range(1, 7)), '|w| =')

That is the whole chapter, arrived at from the learning side.

In [ ]:
print("Five times the training.  The k=4 row is unchanged: still around")
print("80-90% where the window reaches, still 50% where it does not.")
print("The k=9 row, on the same budget, came up across the board.")
print()
print("Training cannot recover information the window never carried.  The")
print("50% is not a model that needs more epochs; it is a model being asked")
print("a question its input does not contain the answer to.")
print()
print("A Turing machine's tape is exactly the memory with no k in it.  That")
print("is why Concept 9's DTM decides this language for every w, and why no")
print("choice of window does.")

## 4. Exercises


1. Predict the `k=5` row, all six columns, then run `train_at(seq, 5)` and check.
2. The floored entries are 50%, not 33%, even though the vocabulary has three tokens.
   What has the model learned that gets it to 50%?
3. Raise `nmax` to 8 and re-run with `k=9`. Where is the wall now? Does the corpus need
   to grow to keep the measurement honest?
4. Give the `k=4` model `n_embd=128` instead of more iterations. Does width recover the
   floored columns? Should it?
5. The argument shows the copy sits $|w|+1$ back regardless of $j$. Where in that
   derivation is the separator `#` used, and what breaks if you remove it?
6. Chapter 12 measured its failure as probability mass on a forbidden symbol; this
   chapter measures accuracy against a cliff. Why does $w\#w$ admit the sharper
   measurement?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter13-TM/Concept-Context-Window-Is-The-Memory')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')